# ウインチ曳航シミュレーター — v1.0

1100 m ground-start条件から5 canonical caseを比較する。StandardとClubは同じplantに対する異なるSOPであり、Clubはtailwheel hold → taut-cable hold-down to 200 ft → nominal 45° climb + Vw protection → terminal fade → `gamma_ground→0` LEVEL OFFを直接モデル化する。


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Video, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
from winch_launch_simulator.simulation import load_case, run_simulation, save_result
from winch_launch_simulator.visualization import (
    make_winch_launch_animation,
    plot_summary,
    save_gif,
    save_mp4,
)

CASE_PATHS = {
    "prescribed": ROOT / "cases" / "g103a_prescribed_tension.json",
    "secant_standard": ROOT / "cases" / "g103a_dynamic_winch.json",
    "secant_club": ROOT / "cases" / "g103a_club_sop_dynamic_winch.json",
    "fem_standard": ROOT / "cases" / "g103a_dynamic_winch_fem.json",
    "fem_club": ROOT / "cases" / "g103a_club_sop_dynamic_winch_fem.json",
}


def event(result, code):
    return next((item for item in result["events"] if item["code"] == code), None)


def launch_metrics(result):
    history = result["history"]
    liftoff = event(result, "LIFTOFF")
    release = event(result, "PILOT_COMMAND_RELEASE")
    release_index = min(range(len(history["time_s"])), key=lambda i: abs(history["time_s"][i] - release["time"]))
    liftoff_index = min(range(len(history["time_s"])), key=lambda i: abs(history["time_s"][i] - liftoff["time"]))
    warnings = [item["code"] for item in result["events"] if item["severity"] == "warning"]
    stops = [item["code"] for item in result["events"] if item["severity"] == "stop"]
    return {
        "case": result["metadata"]["case_id"],
        "cable": result["metadata"]["cable_model"],
        "liftoff_s": liftoff["time"],
        "release_s": release["time"],
        "release_agl_m": history["agl_m"][release_index],
        "release_eas_m_s": history["eas_m_s"][release_index],
        "max_eas_m_s": max(history["eas_m_s"][liftoff_index : release_index + 1]),
        "max_tension_n": max(history["cable_tension_n"][liftoff_index : release_index + 1]),
        "max_abs_q_deg_s": max(abs(value) for value in history["pitch_rate_deg_s"][liftoff_index : release_index + 1]),
        "min_stall_margin_m_s": min(history["stall_margin_m_s"][liftoff_index : release_index + 1]),
        "max_pitch_deg": max(history["theta_deg"][liftoff_index : release_index + 1]),
        "fem_max_ground_penetration_mm": 1000.0 * max(history["fem_max_ground_penetration_m"]),
        "fem_nodes_removed": result["metadata"]["fem_nodes_removed"],
        "power_reduction_request": event(result, "WINCH_POWER_REDUCTION_REQUEST") is not None,
        "warnings": warnings,
        "stops": stops,
    }


def show_metrics(result):
    metrics = launch_metrics(result)
    for key, value in metrics.items():
        print(f"{key}: {value}")
    print("initialization:", result["metadata"]["initialization"])
    return metrics


## 1. Prescribed tension

最低忠実度モデル。機体の ground roll と liftoff は解くが、曳航張力そのものは外部スケジュールとして与える。索の弾性・分布質量・ウインチ回転運動は解かない。


In [ ]:
prescribed_case = load_case(CASE_PATHS["prescribed"])
prescribed = run_simulation(prescribed_case)
prescribed_metrics = show_metrics(prescribed)


In [ ]:
plot_summary(prescribed)


## 2. Dynamic winch + Secant cable

Standardはtension control + EAS full climb。Clubはtailwheel-load ground roll、taut-cable hold-down、45° nominal full climb、Vw feedback、terminal fade→LEVEL OFFを使う。Secant cableは1100 m equivalent property。


In [ ]:
secant_standard_case = load_case(CASE_PATHS["secant_standard"])
secant_standard = run_simulation(secant_standard_case)
secant_standard_metrics = show_metrics(secant_standard)


In [ ]:
plot_summary(secant_standard)


In [ ]:
secant_club_case = load_case(CASE_PATHS["secant_club"])
secant_club = run_simulation(secant_club_case)
secant_club_metrics = show_metrics(secant_club)


In [ ]:
plot_summary(secant_club)


## 3. Dynamic winch + FEM cable

最高忠実度モデル。1100 m material cable、20-node canonical、分布弾性・質量・重量・空力、cable-ground contact/friction、reel-in/node removalを解く。Club pilot/winch commandはSecant Clubと共通。


In [ ]:
fem_standard_case = load_case(CASE_PATHS["fem_standard"])
fem_standard = run_simulation(fem_standard_case)
fem_standard_metrics = show_metrics(fem_standard)


In [ ]:
plot_summary(fem_standard)


In [ ]:
fem_club_case = load_case(CASE_PATHS["fem_club"])
fem_club = run_simulation(fem_club_case)
fem_club_metrics = show_metrics(fem_club)


In [ ]:
plot_summary(fem_club)


## 4. 5 canonical casesを同じ指標で比較する

この表はfidelity/SOPを含むrepresentative comparisonである。純粋なSecant-vs-FEM model-form比較にはmatched-input testを別に行う。


In [ ]:
comparison = [prescribed_metrics, secant_standard_metrics, secant_club_metrics, fem_standard_metrics, fem_club_metrics]
columns = [
    "case", "liftoff_s", "release_s", "release_agl_m", "release_eas_m_s",
    "max_eas_m_s", "max_tension_n", "min_stall_margin_m_s", "max_abs_q_deg_s",
    "max_pitch_deg", "power_reduction_request", "fem_max_ground_penetration_mm",
    "fem_nodes_removed", "warnings", "stops",
]
print(" | ".join(columns))
print(" | ".join("---" for _ in columns))
for row in comparison:
    print(" | ".join(str(row[column]) for column in columns))


## 5. Ground start / FEM ground contact

initializerのpretension/static residual、aircraft/cable penetration、node removalを確認する。penalty contactなのでzero penetrationそのものは要求しない。


In [ ]:
for result in (fem_standard, fem_club):
    history = result["history"]
    print()
    print(result["metadata"]["case_id"])
    print("initialization:", result["metadata"]["initialization"])
    print("max FEM ground penetration [mm]:", 1000.0 * max(history["fem_max_ground_penetration_m"]))
    print("max FEM ground-contact nodes:", max(history["fem_ground_contact_node_count"]))
    print("max aircraft ground penetration [mm]:", 1000.0 * max(history["aircraft_max_ground_penetration_m"]))


## 6. Pilot phase / Club feedback transitions


In [ ]:
def phase_transitions(result):
    history = result["history"]
    rows = []
    previous = None
    for i, phase in enumerate(history["pilot_phase"]):
        if phase != previous:
            rows.append((history["time_s"][i], history["agl_m"][i], history["eas_m_s"][i], phase))
            previous = phase
    return rows

for result in (prescribed, secant_standard, secant_club, fem_standard, fem_club):
    print()
    print(result["metadata"]["case_id"])
    for row in phase_transitions(result):
        print(row)
    requests = [event for event in result["events"] if event["code"] == "WINCH_POWER_REDUCTION_REQUEST"]
    if requests:
        print("power reduction request:", requests[0]["time"])
        active = next((t for t, flag in zip(result["history"]["time_s"], result["history"]["club_speed_reduction_active"]) if flag), None)
        print("speed reduction active:", active)


## 7. 数値結果を保存する


In [ ]:
for result in (prescribed, secant_standard, secant_club, fem_standard, fem_club):
    print(save_result(result, ROOT / "outputs"))


## 8. アニメーションを作る

`make_winch_launch_animation()` は他の `*_animation.ipynb` と同様に、計算済みの `history` を直接受け取る。FEM caseでは保存済みの各節点位置を結び、Secant / Prescribedではhook→winchの1線分を描く。

動画は常に1×実時間で再生する。`fps` は再生倍率ではなく、シミュレーション実時間1秒あたりの表示frame数である。また、follow / overviewともx-zの物理縮尺を1:1（x 1 m = z 1 m）に保つ。


In [ ]:
animation_result = fem_club
fps = 30

fig, ani, axes = make_winch_launch_animation(
    animation_result["history"],
    fps=fps,
    mode="follow",
)


## 9. MP4 / GIF として保存する

保存APIは他の `*_animation.ipynb` と同じ `save_mp4(ani, path=..., fps=..., dpi=...)` / `save_gif(...)` を使う。1×実時間を維持するため、保存時の `fps` にはアニメーション生成時と同じ値を渡す。


In [ ]:
mp4_path = save_mp4(
    ani,
    path=ROOT / "fig" / "winch_launch_animation.mp4",
    fps=fps,
    dpi=120,
)

print(f"saved: {mp4_path}")
display(Video(str(mp4_path), embed=True))
plt.close(fig)

# GIF が必要な場合
# gif_path = save_gif(
#     ani,
#     path=ROOT / "fig" / "winch_launch_animation.gif",
#     fps=fps,
#     dpi=100,
# )
# print(f"saved: {gif_path}")


## 10. V&V status

Release acceptanceは `validation/reference/canonical_regression.json` と `validation/reference/fem_convergence.json` を参照する。Club launch-power transient固有の時系列は `validation/reference/launch_power_ramp.json` をchange-specific referenceとして保持する。FEMのdt/dt2/dt4と20/22/40-node mesh sensitivityはcontroller searchと分離して記録する。
